# Notebook Annexe — XGBoost (Bonus)

**Projet I2ML — Concrete Compressive Strength (UCI)**  
**Auteurs :** Arnaud & Tim

---

## Objectif

Ce notebook est un **complément bonus** au notebook principal (02_nested_cv_models.ipynb).  
On compare XGBoost à notre meilleur modèle retenu (Gradient Boosting sklearn) en utilisant exactement le même protocole de nested CV.

> XGBoost = implémentation optimisée du Gradient Boosting avec régularisation L1/L2 intégrée et parallélisation native. Dépendance externe () non incluse dans le livrable principal.

---
## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Imports OK')

---
## 1. Chargement du dataset nettoyé

In [ ]:
df = pd.read_csv('../../data/concrete_clean.csv')

X = df.drop(columns=['strength'])
y = df['strength']

print(f'Observations : {len(df)}')
print(f'Features     : {X.shape[1]} → {list(X.columns)}')
print(f'Target       : strength (MPa) — moyenne {y.mean():.2f}, std {y.std():.2f}')

---
## 2. Configuration des boucles CV

Identiques au notebook principal pour permettre la comparaison directe.

In [ ]:
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
outer_cv  = KFold(n_splits=5, shuffle=True, random_state=0)

SCORING_INNER = 'neg_mean_squared_error'
N_JOBS = -1

def compute_metrics(scores, model_name=''):
    rmse = np.sqrt(-scores)
    print(f'=== {model_name} ===')
    print(f'RMSE par fold : {np.round(rmse, 3)}')
    print(f'RMSE moyen    : {rmse.mean():.3f} MPa  (±{rmse.std():.3f})')
    return rmse

print(f'Inner CV : {inner_cv}')
print(f'Outer CV  : {outer_cv}')

---
## 3. Gradient Boosting (référence)

On re-run GB avec les mêmes paramètres que le notebook principal pour avoir une baseline de comparaison dans ce contexte.

In [ ]:
pipe_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(random_state=42))
])

param_grid_gb = {
    'model__n_estimators':  [50, 100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth':     [3, 4, 5],
    'model__subsample':     [0.8, 1.0]
}

search_gb = GridSearchCV(pipe_gb, param_grid_gb, cv=inner_cv, scoring=SCORING_INNER, refit=True, n_jobs=N_JOBS)

print('Entraînement GB en cours...')
t0 = time.time()
scores_gb = cross_val_score(search_gb, X=X, y=y, cv=outer_cv, scoring=SCORING_INNER, n_jobs=N_JOBS)
print(f'Terminé en {time.time()-t0:.1f}s')
rmse_gb = compute_metrics(scores_gb, model_name='Gradient Boosting')

---
## 4. XGBoost

In [ ]:
pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0))
])

param_grid_xgb = {
    'model__n_estimators':  [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth':     [3, 4, 5],
    'model__subsample':     [0.8, 1.0]
}

search_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=inner_cv, scoring=SCORING_INNER, refit=True, n_jobs=N_JOBS)

print('Entraînement XGBoost en cours...')
t0 = time.time()
scores_xgb = cross_val_score(search_xgb, X=X, y=y, cv=outer_cv, scoring=SCORING_INNER, n_jobs=N_JOBS)
print(f'Terminé en {time.time()-t0:.1f}s')
rmse_xgb = compute_metrics(scores_xgb, model_name='XGBoost')

---
## 5. Comparaison directe

In [ ]:
rmse_gb_mean, rmse_gb_std = rmse_gb.mean(), rmse_gb.std()
rmse_xgb_mean, rmse_xgb_std = rmse_xgb.mean(), rmse_xgb.std()

print(f'Gradient Boosting : {rmse_gb_mean:.3f} ± {rmse_gb_std:.3f} MPa')
print(f'XGBoost           : {rmse_xgb_mean:.3f} ± {rmse_xgb_std:.3f} MPa')
print(f'Différence         : {abs(rmse_gb_mean - rmse_xgb_mean):.3f} MPa')

# Boxplot
fig, ax = plt.subplots(figsize=(7, 5))
df_plot = pd.DataFrame({'Gradient Boosting': rmse_gb, 'XGBoost': rmse_xgb})
df_plot.plot(kind='box', ax=ax, patch_artist=True)
ax.set_title('RMSE par fold — GB vs XGBoost', fontsize=13)
ax.set_ylabel('RMSE (MPa)')
plt.tight_layout()
plt.show()

---
## 6. Meilleurs hyperparamètres XGBoost

In [ ]:
search_xgb.fit(X, y)
print(f'Meilleurs HPs XGBoost : {search_xgb.best_params_}')
print(f'RMSE inner CV (approx) : {np.sqrt(-search_xgb.best_score_):.3f} MPa')

---
## 7. Conclusion

XGBoost et Gradient Boosting (sklearn) donnent des résultats **statistiquement équivalents** sur ce dataset. Les écarts-types des RMSE par fold se chevauchent — il n'y a pas de différence significative.

**Pourquoi on garde GB comme modèle principal :**
- Pas de dépendance externe ( n'est pas dans la stdlib sklearn)
- Performance identique à la marge de variance
- Plus simple à déployer et à auditer

XGBoost pourrait être préféré dans un contexte production où la vitesse d'inférence compte (parallélisation native), mais pour ce dataset de ~1 000 observations, l'avantage est négligeable.